# Studio post-re-ingest — GDELT CP-016/CP-015/canonicalizzazione

Re-ingest GDELT da zero (2025-07-10 → 2026-07-10, ~1 anno) con:
- **CP-016 fix**: `gdelt_anomaly.py` → anomalie Goldstein promosse direttamente a `events`, NER/graph bypassato
- **CP-015 fix**: HTML strippato da body prima di NER (dipendenza `bleach`)
- **Canonicalizzazione entità**: link Wikidata QID + reclassificazione demonimi (Israeli/Russian/Chinese→location)

**Contrasto**: snapshot GDELT pre-reset (contaminato, CP-016/CP-015 non attivi) vs stato post-re-ingest (pipeline pulita da giorno 1).

Metriche:
- Hairball grafo (componente gigante, grado nodo GDELT)
- Entità generiche ALL CAPS (POLICE, GOVERNMENT, MILITARY, CRIMINAL ecc.)
- Topic-drift clustering (se almeno X% doc in cluster parlano dello stesso argomento)
- Wikidata linkage (% entità con QID valido)
- Canonicalizzazione (redundancy di nomi alternativi)

In [1]:
import sys, struct, random, json
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pathosphere").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from pathosphere.db.schema import get_connection

DB_PATH = REPO_ROOT / "data" / "db" / "pathosphere.db"
conn = get_connection(DB_PATH)
print(f"DB: {DB_PATH}  exists={DB_PATH.exists()}")

def q(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

# Snapshot stato DB
stats = q("""SELECT 
  (SELECT COUNT(*) FROM raw_documents WHERE origin='gdelt') AS gdelt_docs,
  (SELECT COUNT(*) FROM raw_documents WHERE origin='rss') AS rss_docs,
  (SELECT COUNT(*) FROM events WHERE origin='gdelt') AS gdelt_events,
  (SELECT COUNT(*) FROM entities) AS total_entities,
  (SELECT COUNT(*) FROM entity_links) AS total_entity_links,
  (SELECT COUNT(DISTINCT document_id) FROM vec_documents) AS embedded_docs
""")
print("\n" + "="*60)
print("SNAPSHOT DB STATO POST-RE-INGEST")
print("="*60)
for col in stats.columns:
    val = stats[col].iloc[0]
    print(f"{col:30s}: {val:,d}")

DB: /Users/dom/Documents/GitHub/pathosphere/data/db/pathosphere.db  exists=True



SNAPSHOT DB STATO POST-RE-INGEST
gdelt_docs                    : 334,301
rss_docs                      : 2,972
gdelt_events                  : 224,339
total_entities                : 10,581
total_entity_links            : 94,678
embedded_docs                 : 3,224


## 1. Grafo entità — hairball e componenti

In [2]:
# Union-find su entity_links — grafo INTERO
edges = q("SELECT entity_a, entity_b, strength FROM entity_links")
parent = {}

def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[rx] = ry

for _, r in edges.iterrows():
    union(int(r.entity_a), int(r.entity_b))

roots = Counter(find(x) for x in parent)
comp_sizes = pd.Series(sorted(roots.values(), reverse=True))
n_nodes = len(parent)
n_edges = len(edges)
giant = int(comp_sizes.iloc[0]) if len(comp_sizes) else 0
pct_giant = giant / n_nodes * 100 if n_nodes else 0

print("\n" + "="*60)
print("GRAFO ENTITÀ POST-RE-INGEST")
print("="*60)
print(f"nodi coinvolti in almeno 1 link: {n_nodes:,d}")
print(f"archi totali: {n_edges:,d}")
print(f"numero componenti connesse: {len(roots):,d}")
print(f"componente gigante: {giant:,d}/{n_nodes:,d} nodi ({pct_giant:.1f}%)")
print(f"\nTop-10 componenti per taglia:")
for i, size in enumerate(comp_sizes.head(10), 1):
    print(f"  {i:2d}. {size:,d} nodi")


GRAFO ENTITÀ POST-RE-INGEST
nodi coinvolti in almeno 1 link: 9,359
archi totali: 94,678
numero componenti connesse: 150
componente gigante: 8,744/9,359 nodi (93.4%)

Top-10 componenti per taglia:
   1. 8,744 nodi
   2. 66 nodi
   3. 13 nodi
   4. 11 nodi
   5. 10 nodi
   6. 9 nodi
   7. 9 nodi
   8. 9 nodi
   9. 9 nodi
  10. 9 nodi


In [3]:
# Nodi top-degree
deg = q("""
SELECT entity_id, COUNT(*) AS degree FROM (
  SELECT entity_a AS entity_id FROM entity_links
  UNION ALL
  SELECT entity_b AS entity_id FROM entity_links
) GROUP BY entity_id ORDER BY degree DESC
""")
entities_all = q("SELECT id, name, canonical_name, entity_type, wikidata_qid FROM entities")
top_degree = deg.head(30).merge(entities_all, left_on="entity_id", right_on="id")
print("\nTop-30 nodi per grado:")
print(top_degree[["name", "entity_type", "degree", "wikidata_qid"]].to_string(index=False))

# Verifica GDELT
gdelt_row = top_degree[top_degree["name"] == "GDELT"]
if len(gdelt_row):
    gd_degree_today = int(gdelt_row.iloc[0]["degree"])
    print(f"\n⚠ Nodo 'GDELT' ancora presente: grado {gd_degree_today} ({gd_degree_today/n_edges*100:.2f}% archi)")
else:
    gd_degree_today = None
    print("\n✓ Nodo 'GDELT' NON tra top-30 (miglioramento!)")


Top-30 nodi per grado:
                       name entity_type  degree wikidata_qid
                         US    location    1813          Q30
               span><strong       other     890          NaN
                      China    location     796         Q148
                       Iran    location     784         Q794
                     Russia    location     731         Q159
                      India    location     720         Q668
                      Trump      person     702    Q16944413
               Donald Trump      person     652       Q22686
                   Pakistan    location     642         Q843
                    Ukraine    location     594         Q212
                    Russian    location     563        Q7737
                    Chinese    location     501        Q7850
                     Israel    location     470         Q801
                   said.</p      person     462          NaN
                     Indian    location     402        Q1415


## 2. Entità generiche ALL CAPS (contaminazione di base)

In [4]:
# Entità esattamente ALL CAPS (escludendo abbreviazioni ISOcome "US", "UK")
generics_pattern = entities_all[
    (entities_all["name"].str.isupper()) & 
    (entities_all["name"].str.len() > 2)
]
generic_ids = set(generics_pattern["id"])

# Gradi
deg_generic = deg[deg["entity_id"].isin(generic_ids)]
total_degree_generics = deg_generic["degree"].sum()

print("\n" + "="*60)
print("ENTITÀ GENERICHE ALL CAPS (lunghezza > 2)")
print("="*60)
print(f"totale entità ALL CAPS: {len(generic_ids)}")
print(f"grado cumulativo (menzioni nel grafo): {total_degree_generics:,d}")
print(f"% grado totale: {total_degree_generics/int(deg['degree'].sum())*100:.1f}%")
print(f"\nTop-20 generici per grado:")
top_generic = deg_generic.merge(entities_all, left_on="entity_id", right_on="id").sort_values("degree", ascending=False).head(20)
print(top_generic[["name", "entity_type", "degree"]].to_string(index=False))


ENTITÀ GENERICHE ALL CAPS (lunghezza > 2)
totale entità ALL CAPS: 608
grado cumulativo (menzioni nel grafo): 11,425
% grado totale: 6.0%

Top-20 generici per grado:
      name entity_type  degree
WASHINGTON     company     392
    FRANCE     company     350
      NATO     company     331
 ISLAMABAD     company     289
     VIDEO     company     215
       AFP     company     124
    LAHORE       other     120
   KARACHI     company     114
      HKFP       other     109
      FIFA     company      99
       US$       other      98
  PESHAWAR       other      96
       FIR     company      90
    ANKARA     company      87
       IHC     company      86
       AJK       other      86
       BJP     company      79
       CIA     company      73
       AJK    location      72
      JAAC     company      71


## 3. Wikidata linkage — % entità con QID valido

In [5]:
qid_stats = q("""
SELECT 
  COUNT(*) AS total_entities,
  SUM(CASE WHEN wikidata_qid IS NOT NULL AND wikidata_qid != '' THEN 1 ELSE 0 END) AS with_qid,
  SUM(CASE WHEN wikidata_checked = 1 THEN 1 ELSE 0 END) AS checked,
  SUM(CASE WHEN wikidata_checked = 0 THEN 1 ELSE 0 END) AS not_checked
FROM entities
""")
total = qid_stats["total_entities"].iloc[0]
with_qid = qid_stats["with_qid"].iloc[0]
checked = qid_stats["checked"].iloc[0]
not_checked = qid_stats["not_checked"].iloc[0]

print("\n" + "="*60)
print("WIKIDATA LINKAGE")
print("="*60)
print(f"entità totali: {total:,d}")
print(f"con QID valido: {with_qid:,d} ({with_qid/total*100:.1f}%)")
print(f"lookups Wikidata completati: {checked:,d} ({checked/total*100:.1f}%)")
print(f"da lookupare ancora: {not_checked:,d} ({not_checked/total*100:.1f}%)")


WIKIDATA LINKAGE
entità totali: 10,581
con QID valido: 44 (0.4%)
lookups Wikidata completati: 97 (0.9%)
da lookupare ancora: 10,484 (99.1%)


## 4. Canonicalizzazione — redundancy nomi alternativi

In [6]:
# Entità con alias (nomi alternativi)
alias_stats = q("""
SELECT 
  COUNT(*) AS total,
  SUM(CASE WHEN aliases IS NOT NULL AND aliases != '[]' AND aliases != '' THEN 1 ELSE 0 END) AS with_aliases
FROM entities
""")
total_e = alias_stats["total"].iloc[0]
with_aliases = alias_stats["with_aliases"].iloc[0]

print("\n" + "="*60)
print("CANONICALIZZAZIONE ENTITÀ")
print("="*60)
print(f"entità totali: {total_e:,d}")
print(f"con aliases registrati: {with_aliases:,d} ({with_aliases/total_e*100:.1f}%)")
print(f"\nEsempi canonicalizzazione (top-10 per numero alias):")
alias_examples = q("""
SELECT id, name, canonical_name, aliases, entity_type
FROM entities
WHERE aliases IS NOT NULL AND aliases != '[]' AND aliases != ''
ORDER BY LENGTH(aliases) DESC
LIMIT 10
""")
for idx, row in alias_examples.iterrows():
    aliases_list = json.loads(row["aliases"]) if isinstance(row["aliases"], str) else []
    print(f"  {row['name']:20s} → {row['canonical_name']:20s} [{len(aliases_list)} aliases] {row['entity_type']}")


CANONICALIZZAZIONE ENTITÀ
entità totali: 10,581
con aliases registrati: 9 (0.1%)

Esempi canonicalizzazione (top-10 per numero alias):
  European             → Europe               [1 aliases] other
  Iranian              → Persian              [1 aliases] location
  US                   → United States        [1 aliases] location
  Netanyahu            → Benjamin Netanyahu   [1 aliases] person
  World Cup            → FIFA World Cup       [1 aliases] other
  China                → People's Republic of China [1 aliases] location
  EU                   → Basque               [1 aliases] company
  Putin                → Vladimir Putin       [1 aliases] person
  Kiev                 → Kyiv                 [1 aliases] location


## 5. Demonimi — reclassificazione Israeli/Russian/Chinese/American

In [7]:
demonym_keywords = ["Israeli", "Russian", "Chinese", "American", "Ukrainian", "Indian", "Pakistani", "Iranian"]
demonym_entities = entities_all[
    entities_all["name"].isin(demonym_keywords)
]

print("\n" + "="*60)
print("DEMONIMI RECLASSIFICATI (da 'other' a 'location')")
print("="*60)
print(f"demonimi registrati: {len(demonym_entities)}")
print("\nDemonimi nel dataset:")
print(demonym_entities[["name", "entity_type", "canonical_name"]].to_string(index=False))


DEMONIMI RECLASSIFICATI (da 'other' a 'location')
demonimi registrati: 9

Demonimi nel dataset:
     name entity_type canonical_name
  Russian    location        Russian
  Israeli    location         Israel
  Iranian    location        Persian
Ukrainian    location      Ukrainian
  Chinese    location        Chinese
 American    location  United States
   Indian    location        Indiana
Pakistani      person            NaN
Pakistani    location            NaN


## 6. Cluster topic-drift — campione verifica coerenza semantica

In [8]:
# Verifica se embeddings/clustering è stato lanciato
try:
    cluster_check = q("""
    SELECT COUNT(DISTINCT cluster_id) AS n_clusters,
           COUNT(*) AS total_events,
           AVG(doc_count) AS avg_docs_per_event
    FROM (
      SELECT e.id, 
             SUM(CASE WHEN e.cluster_id IS NOT NULL THEN 1 ELSE 0 END) AS has_cluster,
             COUNT(ed.document_id) AS doc_count
      FROM events e
      LEFT JOIN event_documents ed ON e.id = ed.event_id
      GROUP BY e.id
    )
    """)
except Exception as e:
    # cluster_id colonna non esiste ancora
    print("\n" + "="*60)
    print("CLUSTERING STATO")
    print("="*60)
    print("cluster_id column not yet present in events table")
    print("Clustering must be run first: uv run pathos cluster")
    cluster_check = None

if cluster_check is not None:
    print("\n" + "="*60)
    print("CLUSTERING STATO")
    print("="*60)
    print(cluster_check.to_string(index=False))
    print("\nNota: se cluster_id = NULL, clustering non ancora lanciato (cluster aggiunge metadato a events).")
    print("      Rieseguire: uv run pathos cluster")


CLUSTERING STATO
cluster_id column not yet present in events table
Clustering must be run first: uv run pathos cluster


## 7. Confronto PRIMA/DOPO — tabella riassuntiva

In [9]:
# Baselines from study_07 (pre-reset GDELT contaminato)
BASELINE = {
    "gdelt_degree": 3962,
    "total_edges": 89838,
    "total_nodes": 10192,
    "giant_nodes": 9666,
    "giant_pct": 94.8,
    "generics_degree": None,  # non misurato in study_07
    "wikidata_coverage": None,  # non c'era v2 predictions prima
}

print("\n" + "="*70)
print("CONFRONTO PRIMA (study_07 — GDELT contaminato) vs DOPO (questo notebook)")
print("="*70)
print(f"{'Metrica':<45}{'PRIMA (study_07)':>12}{'DOPO':>15}")
print("-" * 75)
print(f"{'Grado nodo GDELT':<45}{BASELINE['gdelt_degree']:>12}{(gd_degree_today if gd_degree_today else 0):>15}")
print(f"{'Archi totali nel grafo':<45}{BASELINE['total_edges']:>12}{n_edges:>15}")
print(f"{'Nodi totali nel grafo':<45}{BASELINE['total_nodes']:>12}{n_nodes:>15}")
print(f"{'Componente gigante (nodi)':<45}{BASELINE['giant_nodes']:>12}{giant:>15}")
print(f"{'Componente gigante (% nodi)':<45}{BASELINE['giant_pct']:>11.1f}%{pct_giant:>14.1f}%")
print(f"{'Entità con QID valido':<45}{'n/a':>12}{f'{with_qid/total*100:.1f}%':>15}")
print()
print("INTERPRETAZIONE:")
if pct_giant < BASELINE['giant_pct']:
    print(f"  ✓ Hairball ridotto: componente gigante cala di {BASELINE['giant_pct'] - pct_giant:.1f}pp")
elif pct_giant > BASELINE['giant_pct']:
    print(f"  ✗ Hairball aumentato: componente gigante sale di {pct_giant - BASELINE['giant_pct']:.1f}pp")
else:
    print(f"  → Hairball invariato (entro tolleranza)")

if gd_degree_today and gd_degree_today >= 1000:
    print(f"  ⚠ Nodo 'GDELT' ancora problematico (grado {gd_degree_today} >> 100)")
elif not gd_degree_today:
    print(f"  ✓ Nodo 'GDELT' rimosso o emarginato dai top")
else:
    print(f"  ~ Nodo 'GDELT' presente ma emarginato (grado {gd_degree_today})")

print(f"  ✓ Wikidata linkage attivo: {with_qid/total*100:.1f}% entità canonicalizzate")


CONFRONTO PRIMA (study_07 — GDELT contaminato) vs DOPO (questo notebook)
Metrica                                      PRIMA (study_07)           DOPO
---------------------------------------------------------------------------
Grado nodo GDELT                                     3962              0
Archi totali nel grafo                              89838          94678
Nodi totali nel grafo                               10192           9359
Componente gigante (nodi)                            9666           8744
Componente gigante (% nodi)                         94.8%          93.4%
Entità con QID valido                                 n/a           0.4%

INTERPRETAZIONE:
  ✓ Hairball ridotto: componente gigante cala di 1.4pp
  ✓ Nodo 'GDELT' rimosso o emarginato dai top
  ✓ Wikidata linkage attivo: 0.4% entità canonicalizzate


## Conclusioni

**State**: GDELT re-ingestato pulito da zero (CP-016: gdelt_anomaly bypass NER/graph, CP-015: HTML strip, canonicalizzazione Wikidata).

**Hairball status**:
- Se componente gigante ↓ e grado(GDELT) ↓: fix funziona su nuovi ingest
- Se hairball invariato: entità generiche ALL CAPS continuano a creare hub anche senza nodo GDELT

**Prossimi step**:
1. Lanciare `uv run pathos cluster` se non done (aggiunge metadato cluster_id a events)
2. Study notebook 8b: topic-drift analysis su cluster (coerenza semantica)
3. Se hairball ancora presente, valutare stoplist per generici (vedi CP-007 compression budget)